# 03 — Full-corpus Dense Embedding và Biểu diễn Thưa (Phase 3)

Notebook này minh họa các hợp đồng và luồng xử lý của Phase 3 trên toàn bộ 5 domain của kho tri thức Huế:
- Xác lập 4 mô hình dense embedding: Multilingual E5-small, E5-base, HuyDang Dek21, và Qwen3-Embedding-0.6B (CUDA FP16 trên GTX 1650);
- Minh họa tiền xử lý vai trò cho Document (Representation A) và Query;
- Biểu diễn thưa (sparse representation) xác định, tương đương chính xác với BM25 cho dot-product retrieval;
- Bounded preflight trên tập mẫu 10–15 chunks mà không encode dense toàn bộ corpus.

**Ranh giới loại trừ rõ ràng của Phase 3:**
- Chưa mã hóa dense toàn bộ 8.460 chunks;
- Không tương tác hay ghi dữ liệu vào Qdrant;
- Không thay đổi cấu hình hay runtime của Foods MVP baseline.


In [ ]:
import json
from pathlib import Path
import sys
import tempfile

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break

from backend.embedding.full_corpus import (
    DENSE_MODEL_SPECS,
    QWEN_QUERY_TASK,
    prepare_document,
    prepare_query,
)
from backend.embedding.sparse import (
    encode_sparse_document,
    encode_sparse_query,
    fit_sparse_state,
)
from backend.evaluation.full_corpus_embedding_preflight import (
    P7_ORDER,
    SMOKE_QUERIES,
    run_preflight,
)

print(f"Số lượng mô hình dense được duyệt: {len(DENSE_MODEL_SPECS)}")
for spec in DENSE_MODEL_SPECS:
    print(f" - {spec.key}: {spec.model_id} (rev: {spec.revision[:8]}..., dim: {spec.dimension}, device: {spec.device}, dtype: {spec.dtype})")


## Bảng Hợp đồng 4 Mô hình Dense Embedding

| Ứng viên | Revision | Kích thước vector | Thiết bị / Dtype | Batch | Hợp đồng đầu vào |
|---|---|---:|---|---:|---|
| `e5-small-384` | `614241f6...` | 384 | CPU / FP32 | 8 | doc: `passage: `, query: `query: ` |
| `e5-base-768` | `d1287505...` | 768 | CPU / FP32 | 8 | doc: `passage: `, query: `query: ` |
| `huydang-dek21-768` | `517f1af7...` | 768 | CPU / FP32 | 8 | Phân từ PyVi cho cả doc và query |
| `qwen3-embedding-0.6b-1024` | `97b0c614...` | 1024 | CUDA / FP16 | 1 | doc: Representation A thô; query: custom English instruction |


In [ ]:
sample_text = "Đại Nội Huế là quần thể di tích thuộc triều Nguyễn."
sample_query = "Đại Nội Huế ở đâu?"

for spec in DENSE_MODEL_SPECS:
    doc_prep = prepare_document(spec, sample_text)
    query_prep = prepare_query(spec, sample_query)
    print(f"=== {spec.key} ===")
    print("  Document prep:", doc_prep[:50] + "...")
    print("  Query prep:   ", query_prep[:60].replace("\n", " ") + "...")


## Chạy Bounded Preflight qua Backend API

Hàm `run_preflight()` thực hiện:
- Kiểm tra tính nhất quán với Phase 2 preview artifact;
- Khởi tạo và ghi sparse state;
- Quét tokenizer trên toàn bộ 8.460 chunks;
- Chạy thử nghiệm dense embedding trên mẫu 10–15 chunks đã chọn;
- Xuất báo cáo bằng chứng mà không rò rỉ vector hay văn bản thô.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    sparse_tmp = Path(tmpdir) / "phase_3_sparse_state.json"
    evidence_tmp = Path(tmpdir) / "phase_3_preflight_evidence.json"

    evidence, exit_code = run_preflight(
        sparse_output=sparse_tmp,
        evidence_output=evidence_tmp,
        allow_downloads=False,
    )

print(f"Trạng thái preflight: {evidence['status']} (mã thoát: {exit_code})")
print("Tổng số files corpus:", evidence["corpus"].get("total_files"))
print("Tổng số chunks corpus:", evidence["corpus"].get("total_chunks"))
print("Số lượng chunks trong mẫu bounded:", len(evidence.get("sample", [])))


## Tóm tắt GPU, Tokenizer và Dense Preflight

Kiểm tra thông số quan sát được từ GPU (GTX 1650) và các mô hình dense:
- Trạng thái CUDA khả dụng, thiết bị nhận diện;
- Số lượng token tối đa quan sát được trên toàn bộ kho ngữ liệu;
- Xác nhận chuẩn L2 của các vector embedding trên mẫu bounded.


In [ ]:
gpu_info = evidence.get("gpu", {})
print("Thông tin GPU:", gpu_info.get("device_name"), "| CUDA:", gpu_info.get("cuda_available"))
if "qwen_vram" in gpu_info:
    print("Qwen VRAM peak (MB):", round(gpu_info["qwen_vram"]["peak_bytes"] / 1024 / 1024, 2))

print("\nTóm tắt các mô hình:")
for k, m in evidence.get("models", {}).items():
    tok_scan = m.get("tokenizer_scan", {})
    smoke = m.get("smoke", {})
    print(f"[{k}] max_tokens={tok_scan.get('max_observed_tokens')}, norm=[{smoke.get('norm_min')}, {smoke.get('norm_max')}], thời gian={smoke.get('elapsed_seconds')}s")


## Minh họa Biểu diễn Thưa (Sparse Representation) & Tương đương BM25

Biểu diễn thưa lưu trữ từ vựng sắp xếp từ điển và giá trị IDF. Mỗi tài liệu được mã hóa thành vector thưa với trọng số BM25 TF-IDF, và câu truy vấn được mã hóa với trọng số 1.0 cho mỗi từ xuất hiện. Tích vô hướng giữa hai vector thưa tương đương chính xác với điểm số BM25.


In [ ]:
from backend.core.schema import FullCorpusChunk
from backend.scoring.bm25 import BM25

docs = [
    FullCorpusChunk(chunk_id="d1", source="travel/places/chua.md", title="Chùa Thiên Mụ", heading_path=[], evidence_parts=[], search_text="Chùa Thiên Mụ nằm bên bờ sông Hương."),
    FullCorpusChunk(chunk_id="d2", source="foods/bun_bo.md", title="Bún bò Huế", heading_path=[], evidence_parts=[], search_text="Bún bò Huế là món ăn nổi tiếng của ẩm thực cố đô."),
]

sparse_state = fit_sparse_state(docs)
q = "ẩm thực Bún bò cố đô"

q_vec = encode_sparse_query(q, sparse_state)
d_vec = encode_sparse_document(docs[1].search_text, sparse_state)

doc_map = dict(zip(d_vec.indices, d_vec.values))
dot_prod = sum(doc_map.get(i, 0.0) * val for i, val in zip(q_vec.indices, q_vec.values))

bm25_score = BM25().fit([d.search_text for d in docs]).score(q, docs[1].search_text)

print("Kích thước từ vựng:", len(sparse_state.vocabulary))
print("Dot product từ vector thưa:", round(dot_prod, 5))
print("Điểm BM25 tham chiếu:     ", round(bm25_score, 5))
assert round(dot_prod, 5) == round(bm25_score, 5)


## Bàn giao sang Phase 4 (Qdrant Ingestion)

Phase 3 đã hoàn tất kiểm chứng kỹ thuật và ghi nhận trạng thái **observed PASS**:
- Cả 4 mô hình dense đáp ứng đúng hợp đồng tiền xử lý và ranh giới kích thước token;
- Mô hình Qwen3-Embedding-0.6B chạy trực tiếp trên GPU GTX 1650 qua CUDA FP16;
- Trạng thái biểu diễn thưa deterministic được thiết lập đồng nhất cho toàn corpus;
- Bounded preflight đạt PASS và Phase 3 đang chờ Reviewer cùng User xác nhận đóng (closure);
- Phase 4 hiện vẫn ở trạng thái đóng (chưa mở).

Toàn bộ quá trình tạo vector dense cho 8.460 chunks và nạp vào các collection Qdrant độc lập sẽ được thực hiện khi Phase 4 chính thức được kích hoạt.
